# JAVDL — Google Colab
Clean, ordered test/runner notebook for the javdl branch.

**Run the cells from top to bottom after a fresh Colab runtime.**

**Required Colab Secrets:** TELEGRAM_API_ID, TELEGRAM_API_HASH, TELEGRAM_BOT_TOKEN.

Secrets are loaded at runtime and are never written into the repository.


## 1. Clone the javdl branch


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/Telebotfaroff/javtiful-scraper.git"
BRANCH = "javdl"
WORKDIR = Path("/content/javdl")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
print("Repository:", REPO)
print("Branch:", BRANCH)
print("Working directory:", Path.cwd())


## 2. Install system and Python dependencies


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg git
!python -m pip install -q -r requirements.txt


## 3. Load Telegram credentials from Colab Secrets

In Colab, open Secrets and add the three required values before running this cell.


In [ ]:
from google.colab import userdata

def get_secret(name):
    try:
        return userdata.get(name)
    except Exception as exc:
        raise RuntimeError(f"Unable to read Colab Secret {name!r}: {exc}") from exc

TELEGRAM_API_ID = get_secret("TELEGRAM_API_ID")
TELEGRAM_API_HASH = get_secret("TELEGRAM_API_HASH")
TELEGRAM_BOT_TOKEN = get_secret("TELEGRAM_BOT_TOKEN")

missing = [
    name for name, value in {
        "TELEGRAM_API_ID": TELEGRAM_API_ID,
        "TELEGRAM_API_HASH": TELEGRAM_API_HASH,
        "TELEGRAM_BOT_TOKEN": TELEGRAM_BOT_TOKEN,
    }.items()
    if not value
]

if missing:
    raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))

os.environ.update({
    "TELEGRAM_API_ID": str(TELEGRAM_API_ID),
    "TELEGRAM_API_HASH": str(TELEGRAM_API_HASH),
    "TELEGRAM_BOT_TOKEN": str(TELEGRAM_BOT_TOKEN),
    "TELEGRAM_SESSION": "javdl_colab",
    "DEFAULT_UPLOADER": "gofile",
})

print("Telegram credentials loaded from Colab Secrets.")


## 4. Environment check


In [ ]:
import shutil
import subprocess
import sys

try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu = "No NVIDIA GPU detected"

print("GPU:", gpu or "No NVIDIA GPU detected")
print("FFmpeg:", shutil.which("ffmpeg") or "NOT FOUND")
print("FFprobe:", shutil.which("ffprobe") or "NOT FOUND")
print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())


## 5. Import and syntax test


In [ ]:
!python -m compileall -q app main.py scripts/colab.py

from app.extractor.javtiful import JavtifulExtractor
from app.uploaders.manager import UploadManager

extractor = JavtifulExtractor()
uploads = UploadManager()

print("Extractor:", extractor.name)
print("Upload providers:", uploads.names())


## 6. Test GoFile guest uploader

This uploads a tiny local test file and verifies that GoFile returns a download URL. It does not involve Telegram.


In [ ]:
from pathlib import Path

test_file = Path("/content/javdl_gofile_test.txt")
test_file.write_text("JAVDL GoFile guest-upload test", encoding="utf-8")

gofile = uploads.get("gofile")
try:
    result = gofile.upload(test_file)
    print("Provider:", result["provider"])
    print("URL:", result.get("download_url"))
    print("Verified:", gofile.verify(result))
finally:
    test_file.unlink(missing_ok=True)


## 7. Test Javtiful extraction

Set a Javtiful post URL you are authorized to access. This step only extracts metadata and URLs exposed by the page.


In [ ]:
JAVTIFUL_URL = ""

if not JAVTIFUL_URL.strip():
    print("Set JAVTIFUL_URL and run this cell to test extraction.")
else:
    video = extractor.extract(JAVTIFUL_URL.strip())
    print("Title:", video.title)
    print("Duration:", video.duration)
    print("Thumbnail:", video.thumbnail)
    print("Qualities:", list(video.qualities))
    if not video.qualities:
        raise RuntimeError("No downloadable quality/source URL was exposed by the page.")


## 8. Start the Telegram bot

**Colab/Jupyter note:** do not use app.run(). Colab already has an asyncio event loop. This cell starts Pyrogram asynchronously and waits for it safely.

Run this cell only after the previous checks pass. Stop it with the Colab interrupt/stop control.


In [ ]:
from pyrogram import idle
from app.bot.telegram import create_app

app = create_app()
print("Starting JAVDL Telegram bot…")

await app.start()
print("JAVDL Telegram bot is running. Send a Javtiful URL to the bot.")

try:
    await idle()
finally:
    if app.is_connected:
        await app.stop()
    print("JAVDL Telegram bot stopped.")
